In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import DoubleType
import pyspark.sql.functions as F

# 1. Khởi tạo Spark Session
spark = SparkSession.builder \
    .appName("Task 2 - Delivery Transform Silver") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,io.delta:delta-spark_2.12:3.1.0,mysql:mysql-connector-java:8.0.33") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

# 2. Đọc dữ liệu thô
path = "s3a://lakehouse/delivery_yt.csv"
df = spark.read.csv(path, header=True, inferSchema=True)
print(f"Đã tải xong dữ liệu thô: {df.count()} dòng")

# 3. Xử lý thời gian
# accept_time / delivery_time đã có dạng "MM-dd HH:mm:ss" -> chỉ cần thêm năm 2022
df_clean = df \
    .withColumn("accept_ts",
        F.to_timestamp(F.concat(F.lit("2022-"), F.col("accept_time")), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("delivery_ts",
        F.to_timestamp(F.concat(F.lit("2022-"), F.col("delivery_time")), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("accept_gps_lng", F.col("accept_gps_lng").cast(DoubleType())) \
    .withColumn("accept_gps_lat", F.col("accept_gps_lat").cast(DoubleType())) \
    .withColumn("lng", F.col("lng").cast(DoubleType())) \
    .withColumn("lat", F.col("lat").cast(DoubleType()))

df_clean = df_clean.withColumn(
    "delivery_minutes",
    (F.unix_timestamp("delivery_ts") - F.unix_timestamp("accept_ts")) / 60
)

# 4. Khoảng cách Haversine (km) bằng hàm native của Spark, không dùng UDF
R = 6371.0
dlat = F.radians(F.col("accept_gps_lat") - F.col("lat"))
dlon = F.radians(F.col("accept_gps_lng") - F.col("lng"))
a = F.sin(dlat / 2) ** 2 + \
    F.cos(F.radians(F.col("lat"))) * F.cos(F.radians(F.col("accept_gps_lat"))) * F.sin(dlon / 2) ** 2
df_clean = df_clean.withColumn("distance_km", F.lit(R) * 2 * F.atan2(F.sqrt(a), F.sqrt(1 - a)))

# 5. Làm sạch: bỏ dòng lỗi thay vì fillna để lỗi không bị che
before = df_clean.count()
df_clean = df_clean.dropna(subset=["accept_ts", "delivery_ts",
                                   "accept_gps_lng", "accept_gps_lat",
                                   "lng", "lat", "distance_km"])
df_clean = df_clean.filter(F.col("delivery_minutes") > 0)

df_clean = df_clean.filter(F.col("delivery_minutes") > 0)
df_clean = df_clean.filter((F.col("delivery_minutes") <= 1440) & (F.col("distance_km") <= 50))

final_columns = ["order_id", "region_id", "city", "courier_id",
                 "lng", "lat", "aoi_id", "aoi_type",
                 "accept_ts", "delivery_ts",
                 "accept_gps_lng", "accept_gps_lat",
                 "delivery_gps_lng", "delivery_gps_lat",
                 "distance_km", "delivery_minutes"]
df_final = df_clean.select(*final_columns)

after = df_final.count()
print(f"Số bản ghi: {before} -> {after} (bỏ {before - after} dòng lỗi)")
if after == 0:
    raise ValueError("Silver rỗng - kiểm tra lại định dạng accept_time/delivery_time")

# 6. Lưu Silver
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("s3a://lakehouse/warehouse/delivery_silver")

print("Đã lưu bảng 'delivery_silver' thành công!")
df_final.select("accept_ts", "delivery_ts", "delivery_minutes", "distance_km").show(5, truncate=False)

Đã tải xong dữ liệu thô: 206431 dòng
Số bản ghi: 206431 -> 201000 (bỏ 5431 dòng lỗi)
Đã lưu bảng 'delivery_silver' thành công!
+-------------------+-------------------+----------------+------------------+
|accept_ts          |delivery_ts        |delivery_minutes|distance_km       |
+-------------------+-------------------+----------------+------------------+
|2022-06-18 08:00:00|2022-06-18 16:21:00|501.0           |6.344111451854702 |
|2022-08-14 08:30:00|2022-08-14 15:01:00|391.0           |6.236543432160278 |
|2022-09-01 09:08:00|2022-09-01 15:15:00|367.0           |5.893901361021074 |
|2022-08-25 13:59:00|2022-08-25 18:26:00|267.0           |12.979484211169458|
|2022-07-09 08:57:00|2022-07-09 10:56:00|119.0           |1.7679597268963432|
+-------------------+-------------------+----------------+------------------+
only showing top 5 rows

